# Train the detector on your own games

This notebook teaches the detector to find the **ball, the referees, the players and the hoop** in the pictures
you checked on the Hoops Tracker dashboard ("Model training"). It runs on Google Colab's free graphics card,
because the laptop has none.

**Before you start**

1. On the dashboard, screen **3 · Dataset**, press "Build version N".
2. Screen **4 · Train**: choose the dataset version, what to start from and the length, then press
   **1 · Send dataset to Drive**. (When Google Drive is not connected on the laptop, the screen shows the zip's
   path instead: copy it into your Drive's `Hoops Tracker/training` folder yourself.)
3. Here in Colab: menu **Runtime**, **Change runtime type**, choose **T4 GPU**, Save.
4. Menu **Runtime**, **Run all**. Colab asks for permission to use your Drive: allow it.

Leave this tab open until the last cell has printed its scores. The dashboard's Models screen shows how far it is
(the "Colab" line) and brings the finished model back.

**Privacy.** The zip holds frames of the games, and they show the players. Sending it to Drive means those frames
are kept by Google. This notebook reads them from your Drive and writes only the finished model and a small score
file back; it sends nothing anywhere else. Clear the outputs (menu **Edit**, **Clear all outputs**) before you save
or share a copy of this notebook: an output can contain pictures.

## 1. Settings

You should not need to change these.

- **The picture size, 1280.** The ball is only about 30 pixels wide in a 1920-pixel frame. At the usual size of
  640 it would shrink to about 10 pixels and be lost; at 1280 it stays about 20 pixels wide. The laptop then runs
  this model at 1280 too (the score file tells it to), which takes about four times as long per frame as 640.
- **"Fresh small model"** means `yolo11n.pt`, the smallest official model: the laptop that will use it has no
  graphics card, so the model has to be small to get through a game in reasonable time. "Start from" an earlier
  model of your own instead to carry on from what it already learned.
- **Length.** "Normal" is 60 rounds through all the pictures, "long" 150. Training stops early by itself when the
  check score has not improved for a while, and the best round is kept.

In [ ]:
DRIVE_FOLDER_NAME = "Hoops Tracker"   # the folder in "My Drive" named on the dashboard's Settings screen (training/ and models/ inside it)
PICTURE_SIZE = 1280                   # pictures are shrunk to this width for training and testing (see above)
BATCH = 8                             # pictures looked at together; halve it if Colab says it is out of memory
EPOCHS = {"normal": 60, "long": 150}  # rounds through all the pictures; "Length" on the Models screen chooses which
FRESH_MODEL = "yolo11n.pt"            # the smallest official model, for "Fresh small model" (downloaded by itself)
WORKERS = 2                           # helpers that load pictures while the graphics card trains
MIN_CONDITION_PICTURES = 5            # a score for "dim" or "blur" is only given from this many test pictures

## 2. Connect Google Drive (a window asks you to allow it)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = f"/content/drive/MyDrive/{DRIVE_FOLDER_NAME}"
WORK_FOLDER = "/content/detector_training"      # working space on Colab; it is gone when Colab closes
print("Drive folder:", DRIVE_ROOT)

## 3. Install the training program

Ultralytics is the program that trains YOLO models. The version is the one installed on the laptop, so the model
file is sure to open there.

In [ ]:
%pip install -q ultralytics==8.4.172

## 4. The newest dataset in Drive

Takes the `dataset_v<n>.zip` with the highest number in the `training` folder, and reads `request.json` next to it
(what the Models screen asked for: the model to start from and the length). The number becomes the model's name:
dataset v3 gives `detector_v3.pt`.

In [ ]:
import json
import re
import shutil
import time
import zipfile
from datetime import date, datetime
from pathlib import Path

TRAINING = Path(DRIVE_ROOT) / "training"
MODELS = Path(DRIVE_ROOT) / "models"
MODELS.mkdir(parents=True, exist_ok=True)


def say(state, line):
    """One line about how far this is, for the dashboard's Models screen (training/status.json), and printed."""
    TRAINING.mkdir(parents=True, exist_ok=True)
    (TRAINING / "status.json").write_text(json.dumps(
        {"state": state, "line": line, "updated": datetime.now().isoformat(timespec="seconds")}))
    print(line)


try:
    # Colab only: when a cell stops with an error, the dashboard's status line says so instead of "running" forever.
    def failed(shell, kind, problem, trace, tb_offset=None):
        say("failed", f"Stopped with an error: {kind.__name__}: {problem}")
        shell.showtraceback()
    get_ipython().set_custom_exc((Exception,), failed)
except NameError:
    pass


def version_number(path):
    """dataset_v3.zip -> 3"""
    return int(re.findall(r"\d+", path.stem)[-1])


zips = sorted(TRAINING.glob("dataset_v*.zip"), key=version_number)
assert zips, f"No dataset_v<n>.zip in {TRAINING}. On the dashboard's Models screen press '1 · Send dataset to Drive' first."
dataset_zip = zips[-1]
VERSION = version_number(dataset_zip)

request = json.loads((TRAINING / "request.json").read_text()) if (TRAINING / "request.json").is_file() else {}
BASE = request.get("base") or "fresh"                  # "fresh", or a .pt file name in Drive's models folder
LENGTH = request.get("length") if request.get("length") in EPOCHS else "normal"
if request.get("dataset_version") not in (None, VERSION):
    print(f"Note: the request was for dataset v{request['dataset_version']}, but the newest zip in Drive is v{VERSION}, which is used.")

say("running", f"Unpacking dataset v{VERSION}...")
dataset = Path(WORK_FOLDER) / f"dataset_v{VERSION}"
shutil.rmtree(dataset, ignore_errors=True)
with zipfile.ZipFile(dataset_zip) as archive:
    archive.extractall(dataset)
DATA = dataset / "data.yaml"
version_info = json.loads((dataset / "version.json").read_text()) if (dataset / "version.json").is_file() else {}
# the class names in the order of their numbers, from data.yaml ("  0: Ball")
CLASSES = [line.split(":", 1)[1].strip() for line in DATA.read_text().splitlines() if re.match(r"\s+\d+:", line)]

print(f"Using {dataset_zip.name}: classes {CLASSES}; starting from {BASE}; length {LENGTH}")
print(version_info.get("split", {}).get("explanation", ""))
for part in ("train", "val", "test"):
    pictures = list((dataset / "images" / part).glob("*.jpg"))
    boxes = sum(len(label.read_text().splitlines()) for label in (dataset / "labels" / part).glob("*.txt"))
    print(f"  {part}: {len(pictures)} pictures, {boxes} boxes")

## 5. The variations

Each variation ticked on the Dataset screen becomes one of Ultralytics' own training settings: in every round,
each training picture is also shown changed that way (darker, tilted, mirrored...), so the model learns to cope
with bad conditions it has not seen. A variation that was not ticked is switched off, so the checklist really decides.

In [ ]:
VARIATIONS = {
    "light":  {"hsv_v": 0.4},                 # darker and brighter
    "zoom":   {"scale": 0.5},                 # zoomed out (smaller ball) and in
    "tilt":   {"degrees": 5.0},               # slight tilt
    "mirror": {"fliplr": 0.5},                # mirror left-right
    "colour": {"hsv_h": 0.03, "hsv_s": 0.6},  # colour shift (other kits)
}
OFF = {"hsv_v": 0.0, "scale": 0.0, "degrees": 0.0, "fliplr": 0.0, "hsv_h": 0.0, "hsv_s": 0.0}

chosen = version_info.get("augmentations", list(VARIATIONS))
train_settings = dict(OFF)
for key in chosen:
    train_settings.update(VARIATIONS.get(key, {}))


def motion_blur_on():
    """"Motion blur" is not one of Ultralytics' settings. It comes from the albumentations package, which Ultralytics
    uses for extra picture changes when the package is installed (Colab usually has it): its own small list of
    changes is replaced by motion blur on 3 training pictures in 10. Returns False when the package is not there."""
    try:
        import albumentations as A
        from ultralytics.data import augment
    except ImportError:
        return False
    plain_init = augment.Albumentations.__init__

    def with_motion_blur(self, p=1.0, transforms=None, flip_idx=None):
        plain_init(self, p, transforms, flip_idx)
        self.transform = A.Compose([A.MotionBlur(blur_limit=(3, 9), p=0.3)])
        self.contains_spatial = False       # blur moves no boxes

    augment.Albumentations.__init__ = with_motion_blur
    return True


notes = []
if "blur" in chosen:
    if motion_blur_on():
        print("Motion blur: on (through albumentations)")
    else:
        notes.append("Motion blur was skipped: the albumentations package is not installed here. The other variations still apply.")
        print(notes[-1])
print("Variations:", ", ".join(chosen) or "none", "->", train_settings)

## 6. Train

Training shows the model the `train` pictures again and again, and after each round checks itself on the `val`
pictures. The `test` pictures are not touched here. With a few hundred pictures expect some tens of minutes on the
T4 (not timed yet). The best round is kept as `best.pt`; the dashboard's Colab line shows the round and the time left.

In [ ]:
from ultralytics import YOLO

epochs = EPOCHS[LENGTH]
if BASE == "fresh":
    start, start_words = FRESH_MODEL, "a fresh small model"
else:
    start, start_words = MODELS / BASE, BASE
    assert Path(start).is_file(), (f"The model to start from, {BASE}, is not in {MODELS}. Press '1 · Send dataset to Drive' "
                                   "again on the dashboard (it copies the model there too), or choose 'Fresh small model'.")
say("running", f"Training on dataset v{VERSION} from {start_words}: {epochs} rounds, pictures {PICTURE_SIZE} wide...")

model = YOLO(str(start))
started = time.time()


def report(trainer):
    """After every round: how far, and about how long is left (the dashboard shows this line)."""
    done = trainer.epoch + 1
    left = (time.time() - started) / done * (trainer.epochs - done)
    say("running", f"Round {done} of {trainer.epochs}, about {max(1, round(left / 60))} min left")


model.add_callback("on_train_epoch_end", report)
# patience: stop early when the check score has not improved for this many rounds (the best round is kept)
model.train(data=str(DATA), imgsz=PICTURE_SIZE, epochs=epochs, batch=BATCH, workers=WORKERS, patience=max(10, epochs // 4),
            project=f"{WORK_FOLDER}/runs", name=f"detector_v{VERSION}", exist_ok=True, plots=False, **train_settings)
best = Path(model.trainer.best)
if not best.is_file():
    best = Path(model.trainer.last)
print(f"Best round kept as {best}")

## 7. The honest test, per class and per condition

The best model is tried on the `test` pictures, which it has never seen, and scored in plain terms:

- **Found** (recall): of the things you marked, the share the model found. Low means missed balls or referees.
- **Right** (precision): of the boxes the model drew, the share that really were that thing. Low means false alarms
  (elbows and shoes taken for the ball).

A marked box counts as found when the model drew a box of the same class over it, overlapping by at least half, with
the same confidence cut-off the pipeline uses. The same count is done again on the dim, the blurred and the zoomed-out
test pictures alone (from `conditions.csv`), so the Models screen can say where the model still struggles. If the
test pictures are stretches of the games the model trained on, the dataset says so, and these numbers flatter it.

The model you started from (if it was one from the models folder) is scored the same way, once, so the chart has a
"Before training" bar.

In [ ]:
import csv

TEST_IMAGES = sorted((dataset / "images" / "test").glob("*.jpg"))
# the words the downloaded model uses for our classes, so it can be scored the same way
ALIASES = {"Referee": ["ref"], "Ball": ["basketball"], "Hoop": ["rim", "basket"]}


def marked_boxes(picture):
    """The coach's boxes for one picture: [(class name, x1, y1, x2, y2)] as shares of the picture."""
    label = dataset / "labels" / picture.parent.name / (picture.stem + ".txt")
    boxes = []
    for line in (label.read_text().splitlines() if label.is_file() else []):
        number, cx, cy, w, h = line.split()
        cx, cy, w, h = float(cx), float(cy), float(w), float(h)
        boxes.append((CLASSES[int(number)], cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2))
    return boxes


def overlap(a, b):
    """How much two boxes overlap, 0 to 1 (the shared area over the area of both together)."""
    width = min(a[2], b[2]) - max(a[0], b[0])
    height = min(a[3], b[3]) - max(a[1], b[1])
    if width <= 0 or height <= 0:
        return 0.0
    shared = width * height
    return shared / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - shared)


def score(model_path, pictures, conf=0.25, min_overlap=0.5):
    """Per class: {"recall", "precision", "marked", "found", "drawn"} on these pictures (see the words above)."""
    model = YOLO(str(model_path))
    names = {number: str(name).lower() for number, name in model.names.items()}
    totals = {name: {"marked": 0, "found": 0, "drawn": 0, "right": 0} for name in CLASSES}
    for picture in pictures:
        result = model(str(picture), imgsz=PICTURE_SIZE, conf=conf, verbose=False)[0]
        drawn = list(zip((names[int(c)] for c in result.boxes.cls), result.boxes.xyxyn.cpu().numpy().tolist()))
        truth = marked_boxes(picture)
        for name in CLASSES:
            wanted = [name.lower()] + ALIASES.get(name, [])
            guesses = [box for word, box in drawn if word in wanted]
            marked = [box[1:] for box in truth if box[0] == name]
            totals[name]["marked"] += len(marked)
            totals[name]["drawn"] += len(guesses)
            used = set()
            for box in marked:
                best_fit = max(((overlap(box, guess), index) for index, guess in enumerate(guesses) if index not in used),
                               default=(0.0, None))
                if best_fit[0] >= min_overlap:
                    totals[name]["found"] += 1
                    totals[name]["right"] += 1
                    used.add(best_fit[1])
    return {name: {"recall": round(t["found"] / t["marked"], 3) if t["marked"] else None,
                   "precision": round(t["right"] / t["drawn"], 3) if t["drawn"] else None,
                   "marked": t["marked"], "found": t["found"], "drawn": t["drawn"]} for name, t in totals.items()}


def pictures_where(**wanted):
    """The test pictures filmed under a condition, from conditions.csv: pictures_where(light="dim")."""
    with open(dataset / "conditions.csv", newline="") as f:
        return [dataset / "images" / "test" / row["image"] for row in csv.DictReader(f)
                if row["split"] == "test" and all(row.get(key) == value for key, value in wanted.items())]


def score_everything(model_path):
    """The per-class scores plus "by_condition": the same on the dim, blurred and zoomed-out test pictures alone."""
    scores = score(model_path, TEST_IMAGES)
    scores["by_condition"] = {}
    for condition, pictures in CONDITIONS.items():
        if len(pictures) >= MIN_CONDITION_PICTURES:
            scores["by_condition"][condition] = dict(score(model_path, pictures), pictures=len(pictures))
        else:
            scores["by_condition"][condition] = {"pictures": len(pictures)}      # too few to say anything
    return scores


def percent(value):
    return "—" if value is None else f"{100 * value:.0f}%"


CONDITIONS = {"dim": pictures_where(light="dim"), "blur": pictures_where(blur="true"), "wide": pictures_where(zoom="wide")}
say("running", f"Scoring on the {len(TEST_IMAGES)} test pictures...")
scores = score_everything(best)

print(f"\nOn the test pictures. {version_info.get('split', {}).get('explanation', '')}")
for name in CLASSES:
    print(f"  {name:8s} found {percent(scores[name]['recall'])} of {scores[name]['marked']} marked; "
          f"{percent(scores[name]['precision'])} of the boxes it drew were right")
for condition, result in scores["by_condition"].items():
    words = (", ".join(f"{name} {percent(result[name]['recall'])}" for name in CLASSES) if CLASSES[0] in result
             else "too few pictures to score")
    print(f"  {condition:8s} ({result['pictures']} pictures): {words}")

base_card = MODELS / (Path(BASE).stem + ".json")
if BASE != "fresh" and not base_card.is_file():
    say("running", f"Scoring {BASE} the same way, for the \"before training\" bar...")
    base_card.write_text(json.dumps({"scored_on_dataset": VERSION, "classes": CLASSES, "scores": score_everything(MODELS / BASE),
                                     "test": {"pictures": len(TEST_IMAGES), "conf": 0.25, "overlap": 0.5}}, indent=2))
    print(f"Saved {base_card}")

## 8. Save the model to Drive

Copies the model to the Drive folder's `models` as `detector_v<n>.pt`, with `detector_v<n>.json` next to it: when it
was trained, from which dataset, and the scores above in the form the dashboard reads.

In [ ]:
import ultralytics

name = f"detector_v{VERSION}"
count = 2
while (MODELS / f"{name}.pt").is_file():            # trained twice from the same dataset: detector_v3_2
    name = f"detector_v{VERSION}_{count}"
    count += 1
shutil.copyfile(best, MODELS / f"{name}.pt")
card = {
    "trained": date.today().isoformat(),
    "dataset_version": VERSION,
    "base": BASE,
    "epochs": epochs,
    "picture_size": PICTURE_SIZE,
    "classes": CLASSES,
    "augmentations": chosen,
    "notes": notes,
    "scores": scores,
    "test": {"pictures": len(TEST_IMAGES), "test_game": version_info.get("test_game"),
             "by_game": version_info.get("split", {}).get("by_game", False), "conf": 0.25, "overlap": 0.5},
    "ultralytics": ultralytics.__version__,
}
(MODELS / f"{name}.json").write_text(json.dumps(card, indent=2))


def found(name):
    return percent(scores.get(name, {}).get("recall"))


say("done", f"Finished: {name}.pt is in Drive's models folder (found ball {found('Ball')}, referee {found('Referee')}, "
            f"player {found('Player')}, hoop {found('Hoop')}). On the dashboard's Models screen press \"Bring the model back\".")
print(f"Saved {MODELS / (name + '.pt')} and {MODELS / (name + '.json')}")

## What next

1. On the dashboard's **Models** screen the Colab line says "Finished". Press **Bring the model back**: it copies
   `detector_v<n>.pt` and its score file from Drive into the laptop's `models` folder (or copy them there yourself).
2. The model appears in the table with its scores. **Try on a clip** measures its speed on the laptop. **Use this
   one** makes the pipeline find the players (and the ball) with it from the next game on.
3. More checked pictures, from more games and gyms, are the surest way to a better model: check more on
   **2 · Annotate**, build the next version, send it, and run this notebook again.